## 05 - CIBERSORTx Deconvolution

### Aim
* Run CIBERSORTx Fractions mode (LM22 default signature) on the TCGA-PAAD cohort to get an independent, reference-based immune-cell-fraction estimate to cross-validate against XDec (notebook 04) and TIMER (notebook 06). Cross-tool comparison itself lives in the EDA notebook (07), not here.

### Input
* `data/processed/TCGA-PAAD_TPM_raw.tsv` (notebook 02) - linear TPM, `gene_symbol` row index, matches CIBERSORTx's mixture-file format directly. Staged as `data/deconvolution/cibersortx/mixture_TPM.txt` (182 samples).

### Output
* `data/deconvolution/cibersortx/CIBERSORTx_Job1_Adjusted.txt` - per-sample LM22 cell-type fractions (22 subtypes) + `P-value`/`Correlation`/`RMSE` goodness-of-fit columns, 182 samples.
* `data/deconvolution/cibersortx/CIBERSORTx_Job1_stackedChart.pdf` - CIBERSORTx's own per-sample stacked-bar visualization.

### Steps
1. Stage the mixture file.
2. Run CIBERSORTx Fractions (Job1) via the web portal (https://cibersortx.stanford.edu/runcibersortx.php) - the Docker CLI route needs a registered token, and approval was still pending, so the web portal was used instead.
3. Load and inspect the results.

### Result
* **Done.** Job1 completed, 182 samples, matches the TIMER/XDec cohort.
* Confirmed from the output file itself: relative **Fractions** mode (22 LM22 columns sum to 1.0 per sample, std ~1e-16, not Absolute mode), **B-mode batch correction** applied (`_Adjusted` filename suffix), permutations > 0 (non-trivial P-value column present).
* **Not yet confirmed** (no job log was saved, not recoverable from the output file alone): exact signature matrix (assumed LM22 default), exact permutation count, quantile-normalization setting.

### Conclusion
* Output is in hand and used in the cross-tool comparison (notebook 07). The signature-matrix/permutations/QN settings above should be filled in once confirmed.

In [1]:
import os
import shutil

import pandas as pd

if os.path.basename(os.getcwd()) == "rnaseq_download":
    os.chdir(os.path.join("..", ".."))
os.getcwd()

'/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer/progress'

### Step 1: stage the mixture file

In [2]:
ciber_dir = "data/deconvolution/cibersortx"
os.makedirs(ciber_dir, exist_ok=True)

mixture_path = os.path.join(ciber_dir, "mixture_TPM.txt")
if not os.path.exists(mixture_path):
    shutil.copy("data/processed/TCGA-PAAD_TPM_raw.tsv", mixture_path)

print("Mixture file ready at:", mixture_path)
with open(mixture_path) as fh:
    print(fh.readline()[:200])

Mixture file ready at: data/deconvolution/cibersortx/mixture_TPM.txt
gene_symbol	TCGA-IB-7888-01A-11R-2156-07	TCGA-FB-A78T-01A-12R-A32O-07	TCGA-2J-AABI-01A-12R-A41B-07	TCGA-HZ-7918-01A-11R-2156-07	TCGA-2J-AABO-01A-21R-A41B-07	TCGA-IB-AAUN-01A-12R-A38C-07	TCGA-HZ-7924-0


### Step 2: run CIBERSORTx Fractions (Job1) - DONE, via the web portal

Run through the CIBERSORTx web portal (https://cibersortx.stanford.edu/runcibersortx.php) - the Docker CLI route needs a registered token, and that token request's approval was still pending, so the mixture file was uploaded directly to the website instead of waiting.

Confirmed from the Job1 output itself: B-mode batch correction applied, relative Fractions mode (not Absolute), permutations > 0 (see Result section above). Signature matrix (assumed LM22 default), exact permutation count, and quantile-normalization setting are not recoverable from the output file and still need confirming.

### Step 3: load results

In [ ]:
results_path = os.path.join(ciber_dir, "CIBERSORTx_Job1_Adjusted.txt")
ciber_results = pd.read_csv(results_path, sep="\t")
print(f"{ciber_results.shape[0]} samples x {ciber_results.shape[1]} columns")
display(ciber_results.head())